[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/cadquery_basics/blob/master/docs/tuto_colab/2d_to_3d.ipynb)

# 3D a partir de 2D
## CadQuery para Arquitetos e Engenheiros
### Versão Google Colab

Este notebook apresenta exemplos de construção de sólidos utilizando polígonos, curvas e as operações `extrude`, `sweep` e `loft` da biblioteca CadQuery. A visualização é feita com o `cadquery_simple_viewer`.

## Instalação

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("Running in Colab, installing packages...")
    !pip install cadquery cadquery-simpleviewer
else:
    print("Not running in Colab, skipping package installation.")

## Importação dos pacotes

In [ ]:
# Instalação (se necessário - descomente caso os pacotes não estejam disponíveis)
# !pip install cadquery cadquery-simple-viewer

import cadquery as cq
from cadquery_simpleviewer import show
from copy import copy

## 1. Extrude – Extrusão de um polígono

Criamos um hexágono regular e extrudamos para formar um prisma hexagonal.

In [ ]:
# Definir o polígono (hexágono) no plano XY
hexagon = cq.Workplane("XY").polygon(6, 10)  # 6 lados, raio 10

# Extrudar para 15 mm de altura
extruded_hex = hexagon.extrude(15)

# Exibir o sólido
show(extruded_hex)

## 2. Sweep – Varredura de um perfil ao longo de uma curva

Criamos um perfil circular e o varremos ao longo de uma curva (spline) definida por pontos.

In [ ]:
# Perfil: círculo de raio 2
profile = cq.Workplane("XY").circle(2)

# Caminho: curva spline passando pelos pontos (0,0,0), (5,5,5), (10,0,10)
path = cq.Workplane("XY").spline([(0,0,0), (5,5,5), (10,0,10)])

# Varredura do perfil ao longo do caminho
swept = profile.sweep(path, isFrenet=True)

# Exibir o resultado
show(swept)

## 3. Loft – Loft entre dois polígonos

Criamos um quadrado na base e um triângulo no topo, e realizamos um loft entre eles.

In [ ]:
# Perfil inferior: quadrado de lado 10
bottom = cq.Workplane("XY").rect(10, 10)

# Perfil superior: triângulo equilátero (polígono de 3 lados) deslocado em Z
top = cq.Workplane("XY", origin=(0,0,15)).polygon(3, 6)  # raio 6, centro no mesmo XY

# Loft entre os dois perfis
lofted = cq.Workplane("XY").add(bottom).add(top).toPending().loft()

# Exibir
show([ top, bottom, lofted],
     
     tessellation_tolerance=.001)

## 4. Combinação – Extrusão de polígono com furo e sweep com perfil poligonal

Exemplo mais avançado: criamos uma base extrusada com um furo e depois fazemos um sweep com um perfil poligonal (pentágono) ao longo de um arco.

In [ ]:
# Base: um retângulo extrudado com furo circular
base = cq.Workplane("XY").rect(30, 20).extrude(5)
base = base.faces(">Z").workplane().circle(5).cutThruAll()

# Caminho para sweep: arco de 90 graus no plano XZ
path_arc = cq.Workplane("XZ",origin=(0,5,0)).radiusArc( (10, 10), 90)

# Perfil: pentágono regular
poly_profile = cq.Workplane("XY").polygon(5, 2.5)

# Sweep do pentágono ao longo do arco
swept_poly = poly_profile.sweep(path_arc, isFrenet=True)

# Combinar a base com o sweep (unir os sólidos)
result = base.union(swept_poly)

# Exibir o resultado final
show(result)

## 5. Loft com múltiplas seções e curvas

Demonstração de loft entre três polígonos diferentes: um quadrado, um octógono e um círculo, com alturas variadas.

In [ ]:
# Seção inferior: quadrado de lado 8
s1 = cq.Workplane("XY").rect(8, 8)

# Seção intermediária: octógono regular, raio 6, na altura 10
s2 = cq.Workplane("XY", origin=(0,0,10)).polygon(8, 6)

# Seção superior: círculo de raio 5, na altura 20
s3 = cq.Workplane("XY", origin=(0,0,20)).circle(5)

# Loft entre as três seções
multi_loft = cq.Workplane("XY").add(s1).add(s2).add(s3).toPending().loft()

# Exibir
show(multi_loft)